In [3]:
# ============================================================
# CELL 1 — SETUP + DATA
# Mount Drive, merge 4 uploaded train QA files,
# and load the fixed test questions from Govt_Chatbot/RAG
# ============================================================

!pip install -q -U \
    unsloth transformers trl datasets accelerate \
    peft bitsandbytes huggingface_hub \
    sacrebleu rapidfuzz bert-score==0.3.13

from google.colab import drive
drive.mount("/content/drive")

import os, glob, json, re, gc, unicodedata
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# ------------------------------------------------------------
# Project folders
# ------------------------------------------------------------

possible = [
    Path("/content/drive/MyDrive/Govt_Chatbot"),
    Path("/content/drive/MyDrive/Govt_Chatbots")
]

BASE = None

for p in possible:
    if (p / "RAG" / "test_questions.csv").exists():
        BASE = p
        break

if BASE is None:
    raise FileNotFoundError(
        "Govt_Chatbot/RAG/test_questions.csv not found."
    )

RAG_DIR = BASE / "RAG"
OUT = BASE / "Llama" / "Fine-Tuned"
MODEL_DIR = OUT / "model_adapter"

OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Find the 4 train JSON files uploaded in /content
# ------------------------------------------------------------

def find_file(patterns):

    files = []

    for pattern in patterns:
        files.extend(glob.glob("/content/" + pattern))

    if not files:
        raise FileNotFoundError(str(patterns))

    return max(files, key=os.path.getmtime)


TRAIN_FILES = {
    "passport": find_file([
        "passport_qa_train*.json"
    ]),

    "nid": find_file([
        "nid_qa_train*.json",
        "NID_qa_train*.json"
    ]),

    "tin": find_file([
        "TIN_qa_train*.json",
        "tin_qa_train*.json"
    ]),

    "birth_death": find_file([
        "birth_death_qa_train*.json"
    ])
}


def norm_domain(x):

    x = str(x).lower()

    if "passport" in x:
        return "passport"
    if "birth" in x or "death" in x:
        return "birth_death"
    if "nid" in x:
        return "nid"
    if "tin" in x:
        return "tin"

    return x


# ------------------------------------------------------------
# Merge training data
# ------------------------------------------------------------

train_rows = []
seen = set()

for domain, path in TRAIN_FILES.items():

    with open(path, encoding="utf-8") as f:
        rows = json.load(f)

    for row in rows:

        instruction = str(
            row.get("instruction", "")
        ).strip()

        input_text = str(
            row.get("input", "")
        ).strip()

        output = str(
            row.get("output", "")
        ).strip()

        d = norm_domain(
            row.get("domain", domain)
        )

        key = (
            str(row.get("id", "")),
            d,
            instruction
        )

        if instruction and output and key not in seen:

            train_rows.append({
                "id": str(row.get("id", "")),
                "domain": d,
                "instruction": instruction,
                "input": input_text,
                "output": output
            })

            seen.add(key)


# Save exact train data used
with open(
    OUT / "unified_train_used.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        train_rows,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# Load fixed test questions from RAG
# ------------------------------------------------------------

tests = pd.read_csv(
    RAG_DIR / "test_questions.csv"
).fillna("")

tests["domain"] = tests["domain"].apply(norm_domain)

assert "question" in tests.columns
assert "gold" in tests.columns
assert (tests["gold"].astype(str).str.strip() != "").all()


print("Training QA:", len(train_rows))
print("Test QA:", len(tests))

print("\nTrain files:")
for d, p in TRAIN_FILES.items():
    print(d, "->", os.path.basename(p))

print("\nOutput folder:", OUT)
print("Model adapter:", MODEL_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training QA: 938
Test QA: 248

Train files:
passport -> passport_qa_train.json
nid -> nid_qa_train.json
tin -> TIN_qa_train.json
birth_death -> birth_death_qa_train.json

Output folder: /content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned
Model adapter: /content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/model_adapter


In [5]:
# ============================================================
# CELL 2 — LLAMA-3.1-8B QLORA FINE-TUNING
# Same training technique/settings as the Qwen notebook
# Saves LoRA adapter permanently to Drive
# ============================================================

from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from huggingface_hub import notebook_login

notebook_login()

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
MAX_SEQ_LENGTH = 1024


# ------------------------------------------------------------
# Load Llama in 4-bit
# ------------------------------------------------------------

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)


# ------------------------------------------------------------
# Same LoRA configuration as Qwen
# ------------------------------------------------------------

model = FastLanguageModel.get_peft_model(
    model,

    r=16,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    lora_alpha=16,
    lora_dropout=0,
    bias="none",

    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


# ------------------------------------------------------------
# Same system prompt used in Qwen fine-tuning
# ------------------------------------------------------------

SYSTEM_PROMPT = (
    "তুমি বাংলাদেশ সরকারের সরকারি সেবা সম্পর্কিত একজন সহায়ক সহকারী। "
    "শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। "
    "যদি তথ্য জানা না থাকে, বলবে 'আমি জানি না'।"
)


# ------------------------------------------------------------
# Format training data using Llama's own chat template
# ------------------------------------------------------------

def formatting(example):

    instruction = str(
        example["instruction"]
    ).strip()

    input_text = str(
        example.get("input", "")
    ).strip()

    answer = str(
        example["output"]
    ).strip()

    if input_text:
        user_text = instruction + "\n" + input_text
    else:
        user_text = instruction

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_text
        },
        {
            "role": "assistant",
            "content": answer
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


dataset = Dataset.from_list(train_rows)

dataset = dataset.map(
    formatting,
    remove_columns=dataset.column_names
)

print("Training examples:", len(dataset))
print("\nExample:\n")
print(dataset[0]["text"][:1000])


# ------------------------------------------------------------
# Same Qwen training settings
# ------------------------------------------------------------

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,

    train_dataset=dataset,
    dataset_text_field="text",

    max_seq_length=MAX_SEQ_LENGTH,

    # Same as Qwen notebook
    packing=True,

    args=SFTConfig(

        output_dir=str(
            OUT / "checkpoints"
        ),

        # Training
        num_train_epochs=3,

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        # Optimizer
        learning_rate=2e-4,
        optim="adamw_8bit",

        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,

        # Precision
        fp16=False,
        bf16=torch.cuda.is_bf16_supported(),

        # Logging / saving
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=3,

        report_to="none",

        seed=3407,
    ),
)


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

train_result = trainer.train()


# ------------------------------------------------------------
# Save LoRA adapter + tokenizer to Drive
# This is what you will use later for RAG + Fine-Tuned
# ------------------------------------------------------------

model.save_pretrained(
    str(MODEL_DIR)
)

tokenizer.save_pretrained(
    str(MODEL_DIR)
)


# Save training log
pd.DataFrame(
    trainer.state.log_history
).to_csv(
    OUT / "training_log.csv",
    index=False
)


# Save experiment configuration
config = {
    "base_model": MODEL_NAME,
    "method": "QLoRA with Unsloth",
    "max_seq_length": 1024,

    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0,

    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    "epochs": 3,
    "batch_size": 2,
    "gradient_accumulation_steps": 4,

    "learning_rate": 2e-4,
    "optimizer": "adamw_8bit",
    "scheduler": "cosine",
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,

    "packing": True,
    "seed": 3407
}

with open(
    OUT / "training_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\nFine-tuning complete.")
print("Adapter saved to:", MODEL_DIR)

==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/938 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 428 | Num Epochs = 3 | Total steps = 162
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,0.769951
20,0.308325
30,0.260250
40,0.239562
50,0.211625
60,0.184871
70,0.167741
80,0.145182
90,0.143378
100,0.138180


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/checkpoints/checkpoint-54/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/checkpoints/checkpoint-108/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/checkpoints/checkpoint-162/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/model_adapter/tokenizer_config.json.



Fine-tuning complete.
Adapter saved to: /content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/model_adapter


In [6]:
# ============================================================
# CELL 3 — FINE-TUNED LLAMA INFERENCE
# Fine-Tuned Llama only: NO RAG
# max_new_tokens = 500
# Saves resumable predictions
# ============================================================

from tqdm.auto import tqdm

MAX_NEW_TOKENS = 500

# Optimize model for inference
FastLanguageModel.for_inference(model)

tokenizer.padding_side = "left"


# ------------------------------------------------------------
# Generate one answer
# ------------------------------------------------------------

def generate_answer(question):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": str(question)
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to("cuda")


    with torch.no_grad():

        outputs = model.generate(
            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=False,

            repetition_penalty=1.05,

            use_cache=True,

            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )


    generated = outputs[0][
        inputs.input_ids.shape[1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and
        generated[-1].item()
        != tokenizer.eos_token_id
    )

    return answer, truncated


# ------------------------------------------------------------
# Resume support
# ------------------------------------------------------------

PARTIAL = OUT / "predictions_partial.csv"

done = {}

if PARTIAL.exists():

    old = pd.read_csv(
        PARTIAL
    ).fillna("")

    for _, row in old.iterrows():

        key = (
            str(row["domain"]),
            str(row["id"])
        )

        done[key] = row.to_dict()


print("Already completed:", len(done))


# ------------------------------------------------------------
# Inference over the fixed test set
# ------------------------------------------------------------

predictions = []

for _, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Fine-Tuned Llama"
):

    key = (
        str(row["domain"]),
        str(row["id"])
    )

    if key in done:

        result = done[key]

    else:

        answer, truncated = generate_answer(
            row["question"]
        )

        result = {
            "id": str(row["id"]),
            "domain": str(row["domain"]),
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "truncated": truncated
        }

        done[key] = result


    predictions.append(result)


    # checkpoint after every answer
    pd.DataFrame(
        predictions
    ).to_csv(
        PARTIAL,
        index=False,
        encoding="utf-8-sig"
    )


# ------------------------------------------------------------
# Final predictions
# ------------------------------------------------------------

pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\nCompleted:", len(pred_df))

print(
    "Truncated:",
    pred_df["truncated"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

print("Predictions saved:")
print(OUT / "predictions.csv")


# Free GPU before BERTScore
del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Already completed: 0


Fine-Tuned Llama:   0%|          | 0/248 [00:00<?, ?it/s]

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


Completed: 248
Truncated: 1
Predictions saved:
/content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/predictions.csv


In [7]:
# ============================================================
# CELL 4 — EVALUATION
# Exact Match, Fuzzy Match, Corpus BLEU,
# ROUGE-1/2/L, Token F1,
# BERT Precision, Recall and F1
# ============================================================

from collections import Counter

from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score


df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

assert (
    df["gold"]
    .astype(str)
    .str.strip()
    != ""
).all(), "Blank gold answer found."


# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ------------------------------------------------------------
# Exact Match
# ------------------------------------------------------------

def exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ------------------------------------------------------------
# Token F1
# ------------------------------------------------------------

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (Counter(p) & Counter(g)).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# ROUGE-N
# ------------------------------------------------------------

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pn = Counter(
        tuple(p[i:i+n])
        for i in range(len(p) - n + 1)
    )

    gn = Counter(
        tuple(g[i:i+n])
        for i in range(len(g) - n + 1)
    )

    overlap = sum(
        (pn & gn).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pn.values())
    recall = overlap / sum(gn.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# ROUGE-L
# ------------------------------------------------------------

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(
                    dp[j - 1] + 1
                )

            else:
                new.append(
                    max(
                        dp[j],
                        new[-1]
                    )
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# Row metrics
# ------------------------------------------------------------

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]


# ------------------------------------------------------------
# Corpus BLEU
# ------------------------------------------------------------

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_bleu = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_bleu = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_bleu,
        [gold_bleu]
    ).score
    / 100
)


# ------------------------------------------------------------
# BERTScore
# ------------------------------------------------------------

print("Calculating BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),

    model_type="bert-base-multilingual-cased",

    batch_size=4,
    device="cpu",

    idf=False,
    rescale_with_baseline=False,

    verbose=True
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


# ------------------------------------------------------------
# Overall result table
# ------------------------------------------------------------

result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1"
    ],

    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean()
    ]
})


# Save predictions with row-level metrics
df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save overall result
result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved artifacts:")
print(OUT / "model_adapter")
print(OUT / "unified_train_used.json")
print(OUT / "training_config.json")
print(OUT / "training_log.csv")
print(OUT / "predictions_partial.csv")
print(OUT / "predictions.csv")
print(OUT / "result.csv")

Calculating BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

computing greedy matching.


  0%|          | 0/62 [00:00<?, ?it/s]

done in 46.50 seconds, 5.33 sentences/sec


,metric,score
0,Exact Match,0.004032
1,Fuzzy Match,0.602077
2,Corpus BLEU,0.134257
3,ROUGE-1,0.319912
4,ROUGE-2,0.162268
5,ROUGE-L,0.285829
6,Token F1,0.319912
7,BERT Precision,0.785074
8,BERT Recall,0.775906
9,BERT F1,0.779315



Saved artifacts:
/content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/model_adapter
/content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/unified_train_used.json
/content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/training_config.json
/content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/training_log.csv
/content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/predictions_partial.csv
/content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/predictions.csv
/content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/result.csv
